# Notebook 1: Tensors & Autograd
**Estimated time:** 30-45 min
**Goal:** Understand PyTorch's fundamental building block — the tensor — and how PyTorch automatically computes gradients using autograd.

---
## What you will learn
1. What a tensor is and how to create one
2. Tensor operations and properties
3. What gradients are and *why* we need them
4. How autograd tracks computations and computes gradients automatically

> **Run every code cell, read every output, then do the project at the end.**

## 1. What is a Tensor?

Imagine you have a single number, like `5`. That's a **scalar**.
Now imagine a row of numbers, like `[1, 2, 3]`. That's a **vector**.
A grid of numbers (rows and columns) is a **matrix**.
A cube of numbers (or higher dimensions) is a **tensor**.

A tensor is just a **container of numbers arranged in a specific shape**.

| Name | Shape example | PyTorch term |
|------|---------------|--------------|
| Scalar | `()` | 0-D tensor |
| Vector | `(3,)` | 1-D tensor |
| Matrix | `(3, 4)` | 2-D tensor |
| Batch of images | `(32, 3, 224, 224)` | 4-D tensor |

PyTorch tensors work like NumPy arrays but with two superpowers:
1. They can live on a **GPU** (much faster math)
2. They can **remember every operation** done to them so gradients can be computed automatically

In [ ]:
import torch
import numpy as np

# --- Creating tensors ---

# From a Python list
a = torch.tensor([1.0, 2.0, 3.0])
print('from list:', a)

# Zeros and ones
zeros = torch.zeros(3, 4)   # 3 rows, 4 cols
ones  = torch.ones(2, 3)
print('zeros shape:', zeros.shape)

# Random numbers (uniform 0-1)
rand  = torch.rand(2, 3)
# Random numbers (normal distribution, mean=0, std=1)
randn = torch.randn(2, 3)
print('rand:\n', rand)

# Sequence (like np.arange)
seq = torch.arange(0, 10, 2)   # start, stop, step
print('arange:', seq)

# From numpy
arr = np.array([4.0, 5.0, 6.0])
from_np = torch.from_numpy(arr)
print('from numpy:', from_np)

In [ ]:
# --- Tensor properties ---

x = torch.randn(3, 4)

print('shape  :', x.shape)        # same as x.size()
print('dtype  :', x.dtype)        # float32 by default
print('device :', x.device)       # cpu or cuda
print('ndim   :', x.ndim)         # number of dimensions
print('numel  :', x.numel())      # total number of elements

## 2. Tensor Operations

Almost everything that works on NumPy arrays works on tensors with the same syntax.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Element-wise arithmetic
print('a + b :', a + b)
print('a * b :', a * b)
print('a ** 2:', a ** 2)

# Reductions
print('sum   :', a.sum())
print('mean  :', a.mean())
print('max   :', a.max())

# Matrix multiplication  (most important op in deep learning!)
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = A @ B          # same as torch.matmul(A, B)
print('matmul shape:', C.shape)   # (3, 5)

# Reshape
x = torch.arange(12, dtype=torch.float32)
print('original:', x.shape)
x = x.reshape(3, 4)
print('reshaped:', x.shape)

# Indexing (same as numpy)
print('first row   :', x[0])
print('last column :', x[:, -1])
print('submatrix   :', x[1:, 1:3])

## 3. What is a Gradient?

**Imagine you're blindfolded on a hilly field and you want to reach the lowest valley.**
You can't see the whole field, but you can feel which direction the ground slopes under your feet.
The **gradient** tells you the slope — which direction is steepest uphill.
To get to the valley (the minimum), you walk **opposite** to the gradient (downhill).

In neural networks:
- The **"field"** is the **loss function** (how wrong our model is)
- The **"position"** is the **model's weights**
- The **"gradient"** tells us how to adjust each weight to reduce the loss
- **"Walking downhill"** = **gradient descent**

PyTorch's **autograd** system tracks every operation on a tensor and automatically computes these gradients for us.

In [ ]:
# --- Autograd basics ---

# To track gradients, set requires_grad=True
x = torch.tensor(3.0, requires_grad=True)

# Do some computation
y = x ** 2 + 2 * x + 1    # y = x^2 + 2x + 1

print('x:', x)
print('y:', y)

# Now ask PyTorch: "what is dy/dx?"
y.backward()               # compute gradients

# dy/dx = 2x + 2, at x=3: 2*3 + 2 = 8
print('gradient dy/dx at x=3:', x.grad)

In [ ]:
# --- Autograd with multiple variables ---

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

# Loss: L = a^2 * b + b^3
L = a**2 * b + b**3

L.backward()

# dL/da = 2ab = 2*2*3 = 12
# dL/db = a^2 + 3b^2 = 4 + 27 = 31
print('dL/da:', a.grad)
print('dL/db:', b.grad)

In [ ]:
# --- Common gotcha: gradients accumulate ---

x = torch.tensor(2.0, requires_grad=True)

for i in range(3):
    y = x * 2
    y.backward()
    print(f'Step {i}: x.grad = {x.grad}')   # accumulates!
    # You MUST zero the gradient before each backward pass:
    x.grad.zero_()

In [ ]:
# --- Detaching from the computation graph ---
# Sometimes you want to use a tensor value without tracking gradients
# (e.g., when logging metrics or doing inference)

x = torch.randn(3, requires_grad=True)
y = x * 2

# Option 1: detach
y_detached = y.detach()
print('detached requires_grad:', y_detached.requires_grad)

# Option 2: torch.no_grad() context (most common during inference)
with torch.no_grad():
    z = x * 2
print('no_grad requires_grad:', z.requires_grad)

## 4. Putting it Together: Manual Gradient Descent

Let's manually implement linear regression using only tensors and autograd.
No `nn.Module`, no optimizer — just raw gradients. This is exactly what PyTorch does internally.

In [ ]:
import matplotlib.pyplot as plt

# True relationship: y = 2x + 1 (we want to learn these weights)
torch.manual_seed(42)
X_data = torch.randn(100, 1)
Y_data = 2 * X_data + 1 + 0.1 * torch.randn(100, 1)   # with noise

# Our learnable parameters (randomly initialized)
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.1
losses = []

for epoch in range(100):
    # 1. Forward pass: compute predictions
    y_pred = X_data * w + b

    # 2. Compute loss (Mean Squared Error)
    loss = ((y_pred - Y_data) ** 2).mean()
    losses.append(loss.item())

    # 3. Backward pass: compute gradients
    loss.backward()

    # 4. Update parameters (gradient descent step)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # 5. Zero gradients before next step!
    w.grad.zero_()
    b.grad.zero_()

print(f'Learned: w={w.item():.4f}, b={b.item():.4f}')
print(f'True:    w=2.0000, b=1.0000')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')

plt.subplot(1, 2, 2)
plt.scatter(X_data.detach(), Y_data.detach(), alpha=0.4, label='Data')
x_line = torch.linspace(-3, 3, 50).unsqueeze(1)
y_line = x_line * w.detach() + b.detach()
plt.plot(x_line, y_line.detach(), 'r-', linewidth=2, label='Learned line')
plt.legend()
plt.title('Fit')
plt.tight_layout()
plt.show()

---
## YOUR TURN — Mini Project

**Task:** Implement gradient descent to fit a **quadratic** function: `y = ax² + bx + c`

**Steps:**
1. Generate data from `y = 3x² - 2x + 1` (add small noise)
2. Initialize parameters `a`, `b`, `c` randomly with `requires_grad=True`
3. Write the training loop (forward → loss → backward → update → zero grad)
4. Train for 200 epochs with `lr=0.05`
5. Print the learned `a`, `b`, `c` vs the true values
6. Plot the training loss

**Hints:**
- Prediction: `y_pred = a * x**2 + b * x + c`
- Loss: MSE (same as above)
- Follow the exact 5-step pattern shown above

Write your code below:

In [ ]:
# YOUR CODE HERE
# 1. Generate data
torch.manual_seed(0)
X = torch.linspace(-3, 3, 100).unsqueeze(1)
# Y = 3*X^2 - 2*X + 1 + noise  <-- fill this in

# 2. Initialize parameters

# 3. Training loop (200 epochs, lr=0.05)

# 4. Print results

# 5. Plot loss